# Heart Disease Prediction - Model Training

This notebook builds a machine learning model that predicts whether a patient has heart disease based on a set of clinical measurements.

**Dataset:** [Heart Disease Dataset](https://www.kaggle.com/datasets/johnsmith88/heart-disease-dataset) (Kaggle, published by *johnsmith88*).

We are running this on Google Colab, so the dataset file downloaded from Kaggle has been uploaded to the Colab session's `/content` folder as `heart.csv`.

## 1. Load the dataset

We read `heart.csv` into a pandas DataFrame and take a first look at the rows and columns.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import json
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

DATA_PATH = "/content/heart.csv"
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
df.head()

## 2. Quick look at the data

We check the data types, look for missing values, and see how balanced the target column is.

In [ ]:
df.info()
print()
print("Missing values per column:")
print(df.isnull().sum())
print()
print("Target counts (0 = no disease, 1 = disease):")
print(df["target"].value_counts())

In [ ]:
df.describe().T

## 3. Split features and target

The `target` column is what we want to predict. Everything else is an input feature.

We split the data into a training set and a testing set. The model learns on the training set and is evaluated on the testing set, which it has never seen before.

In [ ]:
X = df.drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training rows:", X_train.shape[0])
print("Testing rows: ", X_test.shape[0])

## 4. Build the model

We use a scikit-learn `Pipeline`. A pipeline keeps the preprocessing steps (filling missing values and scaling the numbers) bundled together with the classifier. This matters because the API later has to apply the *exact same* steps before predicting.

In [ ]:
model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", RandomForestClassifier(n_estimators=200, random_state=42)),
])

model.fit(X_train, y_train)
print("Model trained.")

## 5. Evaluate on the test set

In [ ]:
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}\n")
print("Classification report:")
print(classification_report(y_test, y_pred, target_names=["No disease", "Disease"]))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

## 6. Save the model and its parameters

We save the trained pipeline as `model.joblib`. We also write the model settings and the test accuracy to `metrics.json` so the results are easy to report and to keep for reference.

In [ ]:
model_dir = Path("/content/model")
model_dir.mkdir(parents=True, exist_ok=True)

model_path = model_dir / "model.joblib"
joblib.dump(model, model_path)

metrics = {
    "dataset": "johnsmith88/heart-disease-dataset",
    "model": "RandomForestClassifier",
    "n_estimators": 200,
    "test_size": 0.2,
    "random_state": 42,
    "accuracy": round(float(accuracy), 4),
    "features": list(X.columns),
}
(model_dir / "metrics.json").write_text(json.dumps(metrics, indent=2))

print("Saved model to:", model_path)
print(json.dumps(metrics, indent=2))

## 7. Test the saved model

As a final check, we load `model.joblib` back from disk and make a prediction on a single patient. This is the same thing the FastAPI service will do when it receives a request.

In [ ]:
loaded_model = joblib.load(model_path)

sample = X_test.iloc[[0]]
prediction = int(loaded_model.predict(sample)[0])
probability = float(loaded_model.predict_proba(sample)[0][1])

print("Sample patient:")
print(sample.to_dict(orient="records")[0])
print()
print(f"Prediction: {prediction} ({'Disease' if prediction == 1 else 'No disease'})")
print(f"Probability of disease: {probability:.4f}")